In [1]:
import glob
import json
from os.path import dirname

import numpy as np

sfi_srs = [1, 2, 4, 8, 16, 32, 64, 128, 256, 384, 640, 960, 1920, 3840]
other_srs = [1]

isruc_path = "../../logs/exp002/exp002a/sweep-2025-07-29_18-44-18_isruc/"

isruc_labels_glob = f"{isruc_path}/*/labels.npz"
isruc_preds_glob = f"{isruc_path}/*/predictions.npz"
isruc_labels_files = sorted(glob.glob(isruc_labels_glob), key=lambda x: int(x.split("/")[-2]))
isruc_preds_files = sorted(glob.glob(isruc_preds_glob), key=lambda x: int(x.split("/")[-2]))

files_per_run_per_sr = {}
for label_file, pred_file in zip(isruc_labels_files, isruc_preds_files):
    with open(dirname(label_file) + "/predict-high-freq.log") as f:
        lines = f.readlines()
        sr_line = [line for line in lines if "sleep_stage_frequency" in line][0]
        sleep_stage_sr = int(sr_line.split("=")[1])
        model_run = [line for line in lines if "model.path=" in line][0]
        model_run = model_run.split("=")[1].strip()
    if model_run not in files_per_run_per_sr:
        files_per_run_per_sr[model_run] = {}
    files_per_run_per_sr[model_run][sleep_stage_sr] = (label_file, pred_file)

In [2]:
lights_events_file = "isruc-info/isruc_lights_events.json"

subj_lights_dict = {}
with open(lights_events_file, "r") as f:
    json_data = json.load(f)
    for s_id in json_data:
        s_id_formatted = "_".join(s_id.split("#")[:-1])
        if s_id_formatted in subj_lights_dict:
            if subj_lights_dict[s_id_formatted]["lights_out"] != json_data[s_id]["l_out"][-1] or \
                    subj_lights_dict[s_id_formatted]["lights_on"] != json_data[s_id]["l_on"][-1]:
                print(f"WARNING: Lights events mismatch for {s_id_formatted}")
        subj_lights_dict[s_id_formatted] = {
            "lights_out": json_data[s_id]["l_out"][-1],
            "lights_on": json_data[s_id]["l_on"][0],
        }
subj_lights_dict

{'isruc-sg2_1_1': {'lights_out': 0, 'lights_on': 932},
 'isruc-sg2_1_2': {'lights_out': 4, 'lights_on': 786},
 'isruc-sg2_7_1': {'lights_out': 2, 'lights_on': 941},
 'isruc-sg2_7_2': {'lights_out': 2, 'lights_on': 898},
 'isruc-sg2_8_1': {'lights_out': 7, 'lights_on': 814},
 'isruc-sg2_8_2': {'lights_out': 6, 'lights_on': 922},
 'isruc-sg2_5_1': {'lights_out': 0, 'lights_on': 814},
 'isruc-sg2_5_2': {'lights_out': 8, 'lights_on': 878},
 'isruc-sg2_3_1': {'lights_out': 4, 'lights_on': 870},
 'isruc-sg2_3_2': {'lights_out': 10, 'lights_on': 812},
 'isruc-sg2_6_1': {'lights_out': 7, 'lights_on': 964},
 'isruc-sg2_6_2': {'lights_out': 4, 'lights_on': 1013},
 'isruc-sg2_2_1': {'lights_out': 0, 'lights_on': 850},
 'isruc-sg2_2_2': {'lights_out': 18, 'lights_on': 868},
 'isruc-sg2_4_1': {'lights_out': 9, 'lights_on': 931},
 'isruc-sg2_4_2': {'lights_out': 12, 'lights_on': 898},
 'isruc-sg3_10': {'lights_out': 1, 'lights_on': 795},
 'isruc-sg3_1': {'lights_out': 0, 'lights_on': 953},
 'isruc-s

In [3]:
def compute_trt_from_epochs(lights_out_epoch: int,
                            lights_on_epoch: int,
                            epoch_length_sec: float = 30.0) -> float:
    """Total recording time (TRT) in minutes."""
    n_epochs = lights_on_epoch - lights_out_epoch
    if n_epochs <= 0:
        raise ValueError("lights_on_epoch must be > lights_out_epoch.")
    return n_epochs * epoch_length_sec / 60.0


def compute_tst_from_hypnogram(hypnogram: np.ndarray,
                               lights_out_epoch: int,
                               lights_on_epoch: int,
                               epoch_length_sec: float = 30.0,
                               sleep_values=None) -> float:
    """Total sleep time (TST) in minutes from a NumPy hypnogram."""
    if sleep_values is None:
        sleep_values = [1, 2, 3, 4]

    # translate lights events into hypnogram sampling rate
    lights_out_epoch = int(lights_out_epoch * 30 / epoch_length_sec)
    lights_on_epoch = int(lights_on_epoch * 30 / epoch_length_sec)
    seg = hypnogram[lights_out_epoch:lights_on_epoch]
    n_sleep_epochs = np.isin(seg, sleep_values).sum()
    return n_sleep_epochs * epoch_length_sec / 60.0


def compute_sleep_latency(hypnogram: np.ndarray,
                          lights_out_epoch: int,
                          epoch_length_sec: float = 30.0,
                          sleep_values=None) -> float:
    """
    Sleep latency (SL) in minutes: lights out → first epoch of any sleep.
    """
    if sleep_values is None:
        sleep_values = [1, 2, 3, 4]

    # translate lights events into hypnogram sampling rate
    lights_out_epoch = int(lights_out_epoch * 30 / epoch_length_sec)
    seg = hypnogram[lights_out_epoch:]
    idx = np.where(np.isin(seg, sleep_values))[0]
    if idx.size == 0:
        raise ValueError("No sleep epochs found.")
    return idx[0] * epoch_length_sec / 60.0


def first_n_rems(arr, n):
    is_rem = (arr == 4).astype(int)

    # Convolve with a window of ones of length n
    # Each position gives the number of rems in that length-n window
    window_sums = np.convolve(is_rem, np.ones(n, dtype=int), mode='valid')

    # Find first index where all n positions are rem (sum == n)
    idx = np.where(window_sums == n)[0]
    return int(idx[0]) if idx.size > 0 else -1


def compute_rem_latency(hypnogram: np.ndarray,
                        lights_out_epoch: int,
                        epoch_length_sec: float = 30.0,
                        sleep_values=None,
                        rem_values=None) -> float:
    """
    REM (Stage R) latency in minutes: sleep onset → first REM epoch.
    """
    if sleep_values is None:
        sleep_values = [1, 2, 3, 4]
    if rem_values is None:
        rem_values = [4]

    # translate lights events into hypnogram sampling rate
    lights_out_epoch = int(lights_out_epoch * 30 / epoch_length_sec)
    # find sleep onset
    seg = hypnogram[lights_out_epoch:]
    sl_idx = np.where(np.isin(seg, sleep_values))[0]
    if sl_idx.size == 0:
        raise ValueError("No sleep epochs found; cannot compute REM latency.")
    sleep_onset_epoch = lights_out_epoch + sl_idx[0]

    # find first REM after sleep onset
    seg_after_so = hypnogram[sleep_onset_epoch:]
    # rem_idx = np.where(np.isin(seg_after_so, rem_values))[0]
    # if rem_idx.size == 0:
    #     raise ValueError("No REM epochs found after sleep onset.")
    # rem_idx = rem_idx[0]
    # find first 5s of continuous REM sleep
    n_cont_idcs = int(np.ceil(5 / epoch_length_sec))
    rem_idx = first_n_rems(seg_after_so, n_cont_idcs)
    if rem_idx == -1:
        raise ValueError("No REM epochs found after sleep onset.")

    return rem_idx * epoch_length_sec / 60.0


def sleep_efficiency(hypnogram: np.ndarray,
                     lights_out_epoch: int,
                     lights_on_epoch: int,
                     epoch_length_sec: float = 30.0,
                     sleep_values=None) -> float:
    """
    Percent sleep efficiency: (TST / TRT) * 100.
    """
    trt = compute_trt_from_epochs(lights_out_epoch, lights_on_epoch, 30)
    tst = compute_tst_from_hypnogram(hypnogram, lights_out_epoch, lights_on_epoch, epoch_length_sec, sleep_values)
    return (tst / trt) * 100.0


def waso(hypnogram: np.ndarray,
         lights_out_epoch: int,
         lights_on_epoch: int,
         epoch_length_sec: float = 30.0,
         sleep_values=None) -> float:
    """
    Wake after sleep onset (WASO) in minutes:
    WASO = TRT − SL − TST
    """
    trt = compute_trt_from_epochs(lights_out_epoch, lights_on_epoch, 30)
    tst = compute_tst_from_hypnogram(hypnogram, lights_out_epoch, lights_on_epoch, epoch_length_sec, sleep_values)
    sl = compute_sleep_latency(hypnogram, lights_out_epoch, epoch_length_sec, sleep_values)
    waso_min = trt - sl - tst
    print(f"{waso_min} = {trt} - {sl} - {tst}, epoch length: {epoch_length_sec} sec")
    if waso_min < 0:
        raise ValueError("Computed WASO is negative; check inputs.")
    return waso_min

In [4]:
se_map = {}
se_gt = {}
for model_run, files_per_sr in files_per_run_per_sr.items():
    print(f"Model: {model_run}")
    se_map[model_run] = {}
    for sleep_stage_sr in other_srs:
        se_map[model_run][sleep_stage_sr] = {}
        sr_data = np.load(files_per_sr[sleep_stage_sr][1])
        sr_data_gt = np.load(files_per_run_per_sr[model_run][1][0])
        for s_id in sr_data:
            gt_ss = sr_data_gt[s_id]
            pred_ss = sr_data[s_id]
            lights_events = subj_lights_dict[s_id]
            gt_art_mask = gt_ss == 9
            if s_id not in se_gt:
                se = sleep_efficiency(gt_ss, lights_events["lights_out"], lights_events["lights_on"], 30)
                se_gt[s_id] = se

            # mark GT artifacts as such in the prediction to remove potential biases
            if sleep_stage_sr == 1:
                pred_ss[gt_art_mask] = 9
            se = sleep_efficiency(pred_ss, lights_events["lights_out"], lights_events["lights_on"], 30 / sleep_stage_sr)
            se_map[model_run][sleep_stage_sr][s_id] = se

Model: anysleep-run1.pth
Model: anysleep-run2.pth
Model: anysleep-run3.pth


In [5]:
waso_map = {}
waso_gt = {}
for model_run, files_per_sr in files_per_run_per_sr.items():
    print(f"Model: {model_run}")
    waso_map[model_run] = {}
    for sleep_stage_sr in other_srs:
        waso_map[model_run][sleep_stage_sr] = {}
        sr_data = np.load(files_per_sr[sleep_stage_sr][1])
        sr_data_gt = np.load(files_per_run_per_sr[model_run][1][0])
        for s_id in sr_data:
            gt_ss = sr_data_gt[s_id]
            pred_ss = sr_data[s_id]
            lights_events = subj_lights_dict[s_id]
            gt_art_mask = gt_ss == 9
            if s_id not in waso_gt:
                w = waso(gt_ss, lights_events["lights_out"], lights_events["lights_on"], 30)
                waso_gt[s_id] = w

            # mark GT artifacts as such in the prediction to remove potential biases
            if sleep_stage_sr == 1:
                pred_ss[gt_art_mask] = 9
            w = waso(pred_ss, lights_events["lights_out"], lights_events["lights_on"], 30 / sleep_stage_sr)
            waso_map[model_run][sleep_stage_sr][s_id] = w

Model: anysleep-run1.pth
126.5 = 439.5 - 5.5 - 307.5, epoch length: 30 sec
122.0 = 439.5 - 4.5 - 313.0, epoch length: 30.0 sec
67.5 = 481.5 - 55.0 - 359.0, epoch length: 30 sec
44.5 = 481.5 - 53.5 - 383.5, epoch length: 30.0 sec
62.5 = 471.0 - 3.5 - 405.0, epoch length: 30 sec
45.5 = 471.0 - 5.5 - 420.0, epoch length: 30.0 sec
13.0 = 481.0 - 1.0 - 467.0, epoch length: 30 sec
10.0 = 481.0 - 0.5 - 470.5, epoch length: 30.0 sec
72.5 = 437.0 - 75.0 - 289.5, epoch length: 30 sec
69.0 = 437.0 - 75.0 - 293.0, epoch length: 30.0 sec
216.0 = 448.0 - 144.5 - 87.5, epoch length: 30 sec
307.5 = 448.0 - 17.5 - 123.0, epoch length: 30.0 sec
63.0 = 466.0 - 4.0 - 399.0, epoch length: 30 sec
37.0 = 466.0 - 2.0 - 427.0, epoch length: 30.0 sec
107.0 = 451.5 - 3.0 - 341.5, epoch length: 30 sec
84.5 = 451.5 - 2.5 - 364.5, epoch length: 30.0 sec
71.5 = 484.0 - 3.0 - 409.5, epoch length: 30 sec
39.0 = 484.0 - 2.0 - 443.0, epoch length: 30.0 sec
116.5 = 420.5 - 43.0 - 261.0, epoch length: 30 sec
103.5 = 420.5

In [6]:
rsl_map = {}
rsl_gt = {}
for model_run, files_per_sr in files_per_run_per_sr.items():
    print(f"Model: {model_run}")
    rsl_map[model_run] = {}
    for sleep_stage_sr in other_srs:
        rsl_map[model_run][sleep_stage_sr] = {}
        sr_data = np.load(files_per_sr[sleep_stage_sr][1])
        sr_data_gt = np.load(files_per_run_per_sr[model_run][1][0])
        for s_id in sr_data:
            gt_ss = sr_data_gt[s_id]
            pred_ss = sr_data[s_id]
            lights_events = subj_lights_dict[s_id]
            gt_art_mask = gt_ss == 9
            if s_id not in rsl_gt:
                try:
                    rsl = compute_rem_latency(gt_ss, lights_events["lights_out"], 30)
                except ValueError:
                    # ignore recordings without REM sleep
                    print(f"WARNING: No REM sleep for {s_id}, sampling rate {sleep_stage_sr}, skipping")
                    continue
                rsl_gt[s_id] = rsl

            # mark GT artifacts as such in the prediction to remove potential biases
            if sleep_stage_sr == 1:
                pred_ss[gt_art_mask] = 9
            try:
                rsl = compute_rem_latency(pred_ss, lights_events["lights_out"], 30 / sleep_stage_sr)
            except ValueError:
                # ignore recordings without REM sleep
                print(f"WARNING: No REM sleep for {s_id}, sampling rate {sleep_stage_sr}, skipping")
                continue
            rsl_map[model_run][sleep_stage_sr][s_id] = rsl

Model: anysleep-run1.pth
Model: anysleep-run2.pth
Model: anysleep-run3.pth


In [7]:
sol_map = {}
sol_gt = {}
for model_run, files_per_sr in files_per_run_per_sr.items():
    print(f"Model: {model_run}")
    sol_map[model_run] = {}
    for sleep_stage_sr in other_srs:
        sol_map[model_run][sleep_stage_sr] = {}
        sr_data = np.load(files_per_sr[sleep_stage_sr][1])
        sr_data_gt = np.load(files_per_run_per_sr[model_run][1][0])
        for s_id in sr_data:
            gt_ss = sr_data_gt[s_id]
            pred_ss = sr_data[s_id]
            lights_events = subj_lights_dict[s_id]
            gt_art_mask = gt_ss == 9
            if s_id not in sol_gt:
                try:
                    sol = compute_sleep_latency(gt_ss, lights_events["lights_out"], 30)
                except ValueError:
                    # ignore recordings without sleep
                    print(f"WARNING: No sleep for {s_id}, sampling rate {sleep_stage_sr}, skipping")
                    continue
                sol_gt[s_id] = sol

            # mark GT artifacts as such in the prediction to remove potential biases
            if sleep_stage_sr == 1:
                pred_ss[gt_art_mask] = 9
            try:
                sol = compute_sleep_latency(pred_ss, lights_events["lights_out"], 30 / sleep_stage_sr)
            except ValueError:
                # ignore recordings without sleep
                print(f"WARNING: No sleep for {s_id}, sampling rate {sleep_stage_sr}, skipping")
                continue
            sol_map[model_run][sleep_stage_sr][s_id] = sol

Model: anysleep-run1.pth
Model: anysleep-run2.pth
Model: anysleep-run3.pth


In [8]:
# save all metrics to file
all_metrics = {
    "se_map": se_map,
    "se_gt": se_gt,
    "waso_map": waso_map,
    "waso_gt": waso_gt,
    "rsl_map": rsl_map,
    "rsl_gt": rsl_gt,
    "sol_map": sol_map,
    "sol_gt": sol_gt,
}
with open("table_s4b_anysleep.json", "w") as f:
    json.dump(all_metrics, f, indent=None)